In [ ]:
import os

num_cores = os.cpu_count()

print(f"Número de núcleos disponíveis: {num_cores}")

Número de núcleos disponíveis: 2


In [ ]:
!cat /proc/cpuinfo

processor	: 0
vendor_id	: GenuineIntel
cpu family	: 6
model		: 79
model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
stepping	: 0
microcode	: 0xffffffff
cpu MHz		: 2199.998
cache size	: 56320 KB
physical id	: 0
siblings	: 2
core id		: 0
cpu cores	: 1
apicid		: 0
initial apicid	: 0
fpu		: yes
fpu_exception	: yes
cpuid level	: 13
wp		: yes
flags		: fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid tsc_known_freq pni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2apic movbe popcnt aes xsave avx f16c rdrand hypervisor lahf_lm abm 3dnowprefetch invpcid_single ssbd ibrs ibpb stibp fsgsbase tsc_adjust bmi1 hle avx2 smep bmi2 erms invpcid rtm rdseed adx smap xsaveopt arat md_clear arch_capabilities
bugs		: cpu_meltdown spectre_v1 spectre_v2 spec_store_bypass l1tf mds swapgs taa mmio_stale_data retbleed
bogomips	: 4399.99
clflush size	: 64
cache_alignment	: 64
addres

In [ ]:
!cat /proc/meminfo

MemTotal:       13290472 kB
MemFree:         9192356 kB
MemAvailable:   12411336 kB
Buffers:          333580 kB
Cached:          3072436 kB
SwapCached:            0 kB
Active:           633280 kB
Inactive:        3240724 kB
Active(anon):       1432 kB
Inactive(anon):   468872 kB
Active(file):     631848 kB
Inactive(file):  2771852 kB
Unevictable:           4 kB
Mlocked:               4 kB
SwapTotal:             0 kB
SwapFree:              0 kB
Dirty:               408 kB
Writeback:             0 kB
AnonPages:        465828 kB
Mapped:           265464 kB
Shmem:              2312 kB
KReclaimable:     112756 kB
Slab:             148872 kB
SReclaimable:     112756 kB
SUnreclaim:        36116 kB
KernelStack:        4400 kB
PageTables:         8704 kB
SecPageTables:         0 kB
NFS_Unstable:          0 kB
Bounce:                0 kB
WritebackTmp:          0 kB
CommitLimit:     6645236 kB
Committed_AS:    2003064 kB
VmallocTotal:   34359738367 kB
VmallocUsed:       10656 kB
VmallocChunk:    

In [ ]:
!lsb_release -a

No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 22.04.3 LTS
Release:	22.04
Codename:	jammy


In [ ]:
%%writefile mmult_seq.c

/*
    This program multiply two N x N dense, square matrices A and B
    to yield the product matrix C = A x B
*/
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <sys/time.h>

// save the marix in a file
void save_matrix(double **matrix, int size){

    char file_name[30];
    sprintf(file_name, "result_matrix.txt");

    // save the result
    FILE *file;
    file = fopen(file_name, "w");

    for(int i = 0; i < size; i++){
        for(int j = 0; j < size; j++){
            fprintf(file, "%5.2f ", matrix[i][j]);
        }
        fprintf(file, "\n");
    }

    fclose(file);
}

int main(int argc, char *argv[]) {

    if(argc != 2){
        printf("Usage: ./mmul_seq N\n");
        printf("N: Matrix size on each axis\n");
        exit(-1);
    }

    // variables to measure execution time
    struct timeval time_start;
    struct timeval time_end;

    int size = atoi(argv[1]);

    // alloc memory for the matrices
    double **a = (double **) malloc(size * sizeof(double *));
    double **b = (double **) malloc(size * sizeof(double *));
    double **c = (double **) malloc(size * sizeof(double *));

    for(int i = 0; i < size; i++){
        a[i] = (double *) malloc(size * sizeof(double));
        b[i] = (double *) malloc(size * sizeof(double));
        c[i] = (double *) malloc(size * sizeof(double));
    }

    // initialize the matrices
    for(int i = 0; i < size; i++){
        for(int j = 0; j < size; j++){
            a[i][j] = i + 1;
            b[i][j] = j + 1;
        }
    }

    // get the start time
    gettimeofday(&time_start, NULL);

    // multiply a x b
    for(int i = 0; i < size; i++){
        for(int j = 0; j < size; j++){

            c[i][j] = 0;

            for(int k = 0; k < size; k++){
                c[i][j] += a[i][k] * b[k][j];
            }
        }
    }
  /*  for (int k=0; k<size; k++)
       for (int i=0; i<size; i++)
       {
          c[i][k] = 0.0;
          for (int j=0; j<size; j++)
             c[i][k] = c[i][k] + a[i][j] * b[j][k];
       }
*/
    // get the end time
    gettimeofday(&time_end, NULL);

    // save the result on a file
    save_matrix(c, size);

    double exec_time = (double) (time_end.tv_sec - time_start.tv_sec) +
                       (double) (time_end.tv_usec - time_start.tv_usec) / 1000000.0;

    printf("Matrices multiplied in %lf seconds\n", exec_time);

    return 0;
}

Overwriting mmult_seq.c


In [ ]:
!gcc -o mmult_seq mmult_seq.c

In [ ]:
!./mmult_seq 1000

Matrices multiplied in 21.900971 seconds


In [ ]:
%%writefile mmult_mpi.c

/*
    This program parallelizes the multiplication of two N x N dense, square matrices A and B
    to yield the product matrix C = A x B using MPI
*/
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <sys/time.h>
#include <mpi.h>

// save the matrix in a file
void save_matrix(double **matrix, int size, int rank) {
    char file_name[30];
    sprintf(file_name, "result_matrix_rank_%d.txt", rank);

    // save the result
    FILE *file;
    file = fopen(file_name, "w");

    for (int i = 0; i < size; i++) {
        for (int j = 0; j < size; j++) {
            fprintf(file, "%5.2f ", matrix[i][j]);
        }
        fprintf(file, "\n");
    }

    fclose(file);
}

int main(int argc, char *argv[]) {
    if (argc != 2) {
        printf("Usage: mpirun -np <num_procs> ./mmul_mpi N\n");
        printf("N: Matrix size on each axis\n");
        exit(-1);
    }

    int size, rank, num_procs;

    MPI_Init(&argc, &argv);
    MPI_Comm_size(MPI_COMM_WORLD, &num_procs);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    if (rank == 0) {
        printf("Number of processes: %d\n", num_procs);
    }

    struct timeval time_start;
    struct timeval time_end;

    size = atoi(argv[1]);

    // Allocate memory for the matrices
    double **a = (double **)malloc(size * sizeof(double *));
    double **b = (double **)malloc(size * sizeof(double *));
    double **c = (double **)malloc(size * sizeof(double *));

    for (int i = 0; i < size; i++) {
        a[i] = (double *)malloc(size * sizeof(double));
        b[i] = (double *)malloc(size * sizeof(double));
        c[i] = (double *)malloc(size * sizeof(double));
    }

    // Initialize the matrices in the root process (rank 0)
    if (rank == 0) {
        for (int i = 0; i < size; i++) {
            for (int j = 0; j < size; j++) {
                a[i][j] = i + 1;
                b[i][j] = j + 1;
            }
        }
    }

    // Broadcast matrices A and B to all processes
    MPI_Bcast(&(a[0][0]), size * size, MPI_DOUBLE, 0, MPI_COMM_WORLD);
    MPI_Bcast(&(b[0][0]), size * size, MPI_DOUBLE, 0, MPI_COMM_WORLD);

    // Get the start time
    MPI_Barrier(MPI_COMM_WORLD);
    gettimeofday(&time_start, NULL);

    // Each process computes its portion of the result matrix
    int chunk_size = size / num_procs;
    for (int i = rank * chunk_size; i < (rank + 1) * chunk_size; i++) {
        for (int j = 0; j < size; j++) {
            c[i][j] = 0;

            for (int k = 0; k < size; k++) {
                c[i][j] += a[i][k] * b[k][j];
            }
        }
    }

    // Gather results back to the root process
    MPI_Gather(&(c[rank * chunk_size][0]), chunk_size * size, MPI_DOUBLE,
               &(c[0][0]), chunk_size * size, MPI_DOUBLE, 0, MPI_COMM_WORLD);

    // Get the end time
    gettimeofday(&time_end, NULL);

    // Save the result in each process
    save_matrix(c, size, rank);

    // Calculate and print the execution time in the root process
    if (rank == 0) {
        double exec_time = (double)(time_end.tv_sec - time_start.tv_sec) +
                           (double)(time_end.tv_usec - time_start.tv_usec) / 1000000.0;

        printf("Matrices multiplied in %lf seconds\n", exec_time);
    }

    MPI_Finalize();

    return 0;
}

Overwriting mmult_mpi.c


In [ ]:
!mpicc -o mmult_mpi mmult_mpi.c

In [ ]:
!mpirun --allow-run-as-root -np 1 ./mmult_mpi 1000

Number of processes: 1
Matrices multiplied in 21.223390 seconds


In [ ]:
!mpirun --allow-run-as-root -np 2 ./mmult_mpi 1000

--------------------------------------------------------------------------
There are not enough slots available in the system to satisfy the 2
slots that were requested by the application:

  ./mmult_mpi

Either request fewer slots for your application, or make more slots
available for use.

A "slot" is the Open MPI term for an allocatable unit where we can
launch a process.  The number of slots available are defined by the
environment in which Open MPI processes are run:

  1. Hostfile, via "slots=N" clauses (N defaults to number of
     processor cores if not provided)
  2. The --host command line parameter, via a ":N" suffix on the
     hostname (N defaults to 1 if not provided)
  3. Resource manager (e.g., SLURM, PBS/Torque, LSF, etc.)
  4. If none of a hostfile, the --host command line parameter, or an
     RM is present, Open MPI defaults to the number of processor cores

In all the above cases, if you want Open MPI to default to the number
of hardware threads instead of the numb

In [ ]:
!mpirun --allow-run-as-root --use-hwthread-cpus -np 2 ./mmult_mpi 1000

Number of processes: 2
Matrices multiplied in 17.781289 seconds


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -np 2 ./mmult_mpi 1000

Number of processes: 2
Matrices multiplied in 17.790284 seconds


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -np 3 ./mmult_mpi 1000

Number of processes: 3
Matrices multiplied in 19.580419 seconds


In [ ]:
!mpirun --allow-run-as-root --oversubscribe  -np 4 ./mmult_mpi 1000

Number of processes: 4
Matrices multiplied in 18.020167 seconds


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -np 8 ./mmult_mpi 1000

Number of processes: 8
Matrices multiplied in 19.580953 seconds
